# AIaaS — MLPerf Training Runner (Standard tier, training)

The **leaderboard-grade training** benchmark, replacing the LoRA/QLoRA *proxy*.
Drives the official **MLPerf Training** reference implementations from the
**`kurtvalcorza/training`** fork — each benchmark trains a reference model to a
**target quality** under MLPerf's rules (the training analog of MLPerf Inference's
accuracy gate).

### Honest scope — read first
MLPerf Training is **cluster-scale**: a benchmark is "train model X on dataset Y to
target accuracy Z", which is **many GPU-hours to days** and assumes full datasets
(ImageNet, C4, COCO, …). On a **single A100 40 GB** most MLPerf Training benchmarks
are **not practically runnable to completion** — the strategy doc itself calls full
MLPerf Training overkill for one 40 GB card. This notebook is therefore a
**portable runner/bridge**: it clones the fork, sets up a chosen benchmark, and can
launch a **short smoke run** (a few steps, to validate the pipeline + measure
throughput); the *credible, to-target* run belongs on adequate (multi-GPU) hardware
in a `training`-fork session.

> If you only need fine-tuning *capacity* numbers on one card, that was the LoRA
> proxy — intentionally dropped here because it isn't industry-comparable. There is
> no standardized LoRA-fine-tune benchmark; MLPerf Training is the comparable
> harness, with the caveats above.

## 1. Clone the training fork

In [ ]:
import os, subprocess, sys, time, json

REPO_URL = "https://github.com/kurtvalcorza/training"
TRAIN = os.path.abspath("training-ref")

def sh(cmd, cwd=None, env=None):
    print("$", " ".join(cmd), f"(cwd={cwd})" if cwd else "")
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-1500:])
    if p.returncode != 0: print("STDERR:", p.stderr[-1500:])
    return p

if not os.path.isdir(TRAIN):
    sh(["git", "clone", "--depth", "1", REPO_URL, TRAIN])
print("benchmarks in fork:")
for d in sorted(os.listdir(TRAIN)):
    if os.path.isdir(os.path.join(TRAIN, d)) and not d.startswith("."):
        print("  ", d)


## 2. Environment & hardware capture

In [ ]:
import platform, datetime, torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": (torch.cuda.get_device_name(0) if CUDA else "cpu"),
    "gpu_count": (torch.cuda.device_count() if CUDA else 0),
    "vram_total_gb": (round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if CUDA else None),
    "cuda": torch.version.cuda, "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__, "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))
if ENV["gpu_count"] < 8:
    print("\nNOTE: MLPerf Training reference configs target multi-GPU nodes; on", 
          ENV["gpu_count"], "GPU(s) run SMOKE only (a to-target run is infeasible here).")


## 3. Configuration
Pick a benchmark directory from the fork (printed above). Default points at the
image-classification (ResNet) reference. `SMOKE=True` runs a handful of steps to
validate the pipeline and measure throughput, not to target accuracy.

In [ ]:
BENCHMARK_DIR = "image_classification"   # e.g. image_classification | language_model | object_detection
SMOKE = True                              # short run (pipeline + throughput); not to-target
OUT_DIR = os.path.abspath("mlperf_training_results")
os.makedirs(OUT_DIR, exist_ok=True)

BPATH = os.path.join(TRAIN, BENCHMARK_DIR)
assert os.path.isdir(BPATH), f"{BENCHMARK_DIR} not found in fork; pick one from the list above"
print("benchmark path:", BPATH)
# Show the reference run instructions so you can set up datasets/launch correctly.
for name in ("README.md", "run_and_time.sh", "run_reference.sh"):
    fp = os.path.join(BPATH, name)
    if os.path.exists(fp):
        print(f"\n----- {name} (head) -----")
        print(open(fp, errors="ignore").read()[:1200])


## 4. Set up the benchmark
MLPerf Training reference impls are typically Docker- and dataset-driven. Install
the benchmark's Python requirements if present; **datasets must be downloaded per
the benchmark README** (ImageNet/C4/COCO — large, often gated).

In [ ]:
req = os.path.join(BPATH, "requirements.txt")
if os.path.exists(req):
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", req])
else:
    print("no requirements.txt at top level; check subdirs / the benchmark README + Dockerfile")

# Point this at your prepared dataset dir (see the benchmark README for the layout).
DATA_DIR = ""   # REQUIRED for a real run
print("Set DATA_DIR to the prepared dataset before launching. DATA_DIR =", DATA_DIR or "(unset)")


## 5. Launch (smoke by default)
This calls the benchmark's reference launch script. Real MLPerf Training is
controlled by that script's env/flags (epochs, target accuracy, world size); for a
smoke run, cap steps/epochs per the benchmark's options. Edit `LAUNCH` to match the
benchmark you chose.

In [ ]:
# Example launch — ADJUST per the benchmark README (script name, flags, env).
# Most reference impls expose run_and_time.sh (often Docker) or a python train.py.
LAUNCH = ["bash", "run_and_time.sh"]      # placeholder; set per benchmark
run_env = {**os.environ}
if DATA_DIR:
    run_env["DATA_DIR"] = DATA_DIR

result = {"status": "not_run"}
script = os.path.join(BPATH, LAUNCH[-1])
if not DATA_DIR:
    print("DATA_DIR unset -> skipping launch (set up the dataset first). "
          "This cell is a no-op until then.")
elif not os.path.exists(script):
    print(f"{LAUNCH[-1]} not found in {BENCHMARK_DIR}; open the README and set LAUNCH "
          "to the correct script/flags for this benchmark.")
else:
    t0 = time.time()
    p = sh(LAUNCH, cwd=BPATH, env=run_env)
    result = {"status": "ok" if p.returncode == 0 else f"exit_{p.returncode}",
              "elapsed_s": round(time.time() - t0, 1),
              "stdout_tail": (p.stdout or "")[-3000:]}


## 6. Parse + normalize
MLPerf Training emits **MLLog** lines (`:::MLLOG ...`) with `init_stop`/`run_start`/
`run_stop`, `eval_accuracy`, and `throughput`. Scrape what's present; the full log
is kept for the official result.

In [ ]:
import re

def parse_mllog(text):
    out = {"throughput": None, "eval_accuracy": None, "run_start": None, "run_stop": None}
    for line in (text or "").splitlines():
        if ":::MLLOG" in line or '"key"' in line:
            for key in ("throughput", "eval_accuracy", "run_start", "run_stop"):
                if f'"{key}"' in line:
                    m = re.search(r'"value"\s*:\s*([-\d.eE]+)', line)
                    if m:
                        try: out[key] = float(m.group(1))
                        except ValueError: pass
    return out

metrics = parse_mllog(result.get("stdout_tail", "")) if result.get("status") == "ok" else {}

NORM = {
    "schema": "mlperf-training/1.0",
    "env": ENV, "benchmark": BENCHMARK_DIR, "smoke": SMOKE,
    "result": result, "metrics": metrics,
}
out = os.path.join(OUT_DIR, f"mlperf_training_{BENCHMARK_DIR}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)
print(json.dumps({"status": result.get("status"), "metrics": metrics}, indent=2))


## 7. Notes — credible runs & automation

- **Datasets** are large and per-benchmark (ImageNet, C4/Wikipedia, COCO, …) — see
  each benchmark's README; some require registration.
- A **to-target / RCP-compliant** MLPerf Training run needs the reference world
  size (often 8+ GPUs) and runs for hours–days. On one 40 GB A100 use this for a
  **smoke / throughput** signal only.
- **Automation:** MLCommons also provides CM/MLCFlow flows for some training
  benchmarks — check the fork + docs for `run-mlperf,training`-style scripts.
- Runs belong in a session scoped to the **`kurtvalcorza/training`** fork (and
  adequate hardware); this notebook is the portable bridge.